# 02 — Scrape Exeter Module Bank: Module Detail Pages

**Purpose.** For each module listed in `index_all.csv`, fetch its descriptor page and extract the structured information we need for the analysis: credits, cohort size, teaching hours, summative assessment breakdown (coursework / written exam / practical %), NQF level, and the number of distinct summative assessment items.

**Source.** `https://www.exeter.ac.uk/study/studyinformation/modules/info?moduleCode=<CODE>&ay=<YYYY/Y>&sys=0`

**Inputs.** `data/raw/module_bank/index_all.csv` from notebook 01.

**Output.** `data/raw/module_bank/modules_details.csv` — one row per (module_code, academic_year).

**Design choices.**
- We save progress **incrementally** to `modules_details.csv`, so re-running the notebook after a crash picks up where it left off and doesn't re-hit already-scraped pages.
- The detail page is a series of small tables rather than one big one; each is identified by the preceding H2/H3 heading. The parser walks the DOM in order and reads each recognisable section into a flat dict.
- Fields that can't be found are stored as `None`; the cleaning notebook will report on completeness.


## 1. Imports and configuration

In [1]:
import os
import re
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- Paths ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "module_bank"
INDEX_PATH = RAW_DIR / "index_all.csv"
DETAILS_PATH = RAW_DIR / "modules_details.csv"

# --- Scraping config ---
DETAIL_URL = "https://www.exeter.ac.uk/study/studyinformation/modules/info"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Exeter BEE2041 student project; educational research)"
}
SLEEP_BETWEEN_REQUESTS = 1.0
REQUEST_TIMEOUT = 20

print("Index file :", INDEX_PATH)
print("Output file:", DETAILS_PATH)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\kenna\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\kenna\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\kenna\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\kenna\anaconda3\Lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\kenna\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\kenna\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\kenna\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\kenna\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Index file : C:\Users\kenna\datasci\files\data\raw\module_bank\index_all.csv
Output file: C:\Users\kenna\datasci\files\data\raw\module_bank\modules_details.csv


## 2. Load the master index and figure out what's already been done

In [3]:
index_df = pd.read_csv(INDEX_PATH)
print(f"Loaded index: {len(index_df):,} (module, year) rows")

# Already-scraped rows (so a re-run resumes rather than redoes work).
if DETAILS_PATH.exists():
    done_df = pd.read_csv(DETAILS_PATH)
    done_keys = set(zip(done_df["module_code"], done_df["academic_year"]))
    print(f"Found existing output: {len(done_df):,} rows already scraped — will skip these.")
else:
    done_keys = set()
    print("No existing output — will scrape from scratch.")

todo = index_df[~index_df.apply(
    lambda r: (r["module_code"], r["academic_year"]) in done_keys, axis=1
)].reset_index(drop=True)
print(f"Remaining to scrape: {len(todo):,} pages")

Loaded index: 9,073 (module, year) rows
No existing output — will scrape from scratch.
Remaining to scrape: 9,073 pages


## 3. The parser

The Exeter module descriptor page is a predictable layout of small tables under named headings. Each table we care about is found by searching for its heading, then walking its `<tr>` rows. A handful of small helpers keep the extraction readable.

In [5]:
def _clean(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def _table_after(soup_root, heading_keywords: list[str]):
    """Find the first <table> that appears after a heading containing all given keywords."""
    for tag in soup_root.find_all(["h2", "h3", "h4"]):
        text = tag.get_text(" ", strip=True).lower()
        if all(kw.lower() in text for kw in heading_keywords):
            sib = tag.find_next("table")
            if sib is not None:
                return sib
    return None


def _first_two_col_table(soup_root):
    """The module's top metadata table (Module title | Economic Principles, etc.)."""
    for t in soup_root.find_all("table"):
        first_row = t.find("tr")
        if not first_row:
            continue
        cells = first_row.find_all(["th", "td"])
        if len(cells) == 2 and "module title" in cells[0].get_text(" ", strip=True).lower():
            return t
    return None


def _kv_from_two_col_table(table):
    """Turn a two-column table into a {label: value} dict."""
    out = {}
    if table is None:
        return out
    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        if len(cells) >= 2:
            k = _clean(cells[0].get_text(" ", strip=True)).lower()
            v = _clean(cells[1].get_text(" ", strip=True))
            out[k] = v
    return out


def _int_or_none(s):
    if s is None:
        return None
    m = re.search(r"-?\d+", str(s))
    return int(m.group(0)) if m else None


def parse_module_page(html: str) -> dict:
    """Parse an Exeter module descriptor page into a flat dict of fields."""
    soup = BeautifulSoup(html, "html.parser")
    main = soup.find(id="main-col") or soup.find("main") or soup

    result = {
        "module_title": None,
        "credits": None,
        "num_students_anticipated": None,
        "scheduled_hours": None,
        "independent_hours": None,
        "placement_hours": None,
        "coursework_pct": None,
        "written_exam_pct": None,
        "practical_exam_pct": None,
        "n_summative_items": None,
        "nqf_level": None,
        "distance_learning": None,
        "origin_date": None,
        "last_revision_date": None,
    }

    # --- 1. Top metadata block ---
    meta = _kv_from_two_col_table(_first_two_col_table(main))
    result["module_title"] = meta.get("module title")
    result["credits"] = _int_or_none(meta.get("credits"))

    # --- 2. Anticipated cohort size ---
    #     Look in any two-col table for the label containing "students taking module".
    for t in main.find_all("table"):
        for tr in t.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) >= 2:
                label = _clean(cells[0].get_text(" ", strip=True)).lower()
                if "students taking module" in label:
                    result["num_students_anticipated"] = _int_or_none(cells[1].get_text(" ", strip=True))

    # --- 3. Learning activities: "Scheduled | Independent | Placement" ---
    t = _table_after(main, ["learning activities"])
    if t is not None:
        rows = t.find_all("tr")
        # Header row tells us the order; data row tells us the numbers.
        if len(rows) >= 2:
            headers = [_clean(c.get_text(" ", strip=True)).lower() for c in rows[0].find_all(["th", "td"])]
            values  = [_clean(c.get_text(" ", strip=True))          for c in rows[1].find_all(["th", "td"])]
            for h, v in zip(headers, values):
                if "scheduled" in h:
                    result["scheduled_hours"] = _int_or_none(v)
                elif "independent" in h:
                    result["independent_hours"] = _int_or_none(v)
                elif "placement" in h or "study abroad" in h:
                    result["placement_hours"] = _int_or_none(v)

    # --- 4. Summative assessment percentages ---
    t = _table_after(main, ["summative assessment"])
    if t is not None:
        rows = t.find_all("tr")
        if len(rows) >= 2:
            headers = [_clean(c.get_text(" ", strip=True)).lower() for c in rows[0].find_all(["th", "td"])]
            values  = [_clean(c.get_text(" ", strip=True))          for c in rows[1].find_all(["th", "td"])]
            for h, v in zip(headers, values):
                if "coursework" in h:
                    result["coursework_pct"] = _int_or_none(v)
                elif "written exam" in h:
                    result["written_exam_pct"] = _int_or_none(v)
                elif "practical" in h:
                    result["practical_exam_pct"] = _int_or_none(v)

    # --- 5. Number of summative assessment items ---
    #     The "Details of summative assessment" table lists each item on a separate row.
    t = _table_after(main, ["details", "summative assessment"])
    if t is not None:
        data_rows = t.find_all("tr")[1:]  # skip header
        # A row counts if its first cell has non-empty text.
        n_items = sum(
            1 for tr in data_rows
            if tr.find(["th", "td"]) and _clean(tr.find(["th", "td"]).get_text(" ", strip=True))
        )
        result["n_summative_items"] = n_items if n_items > 0 else None

    # --- 6. Footer metadata: NQF level, distance learning, dates ---
    for t in main.find_all("table"):
        for tr in t.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) < 2:
                continue
            label = _clean(cells[0].get_text(" ", strip=True)).lower()
            value = _clean(cells[1].get_text(" ", strip=True))
            if "nqf level" in label and result["nqf_level"] is None:
                result["nqf_level"] = _int_or_none(value)
            elif "distance learning" in label and result["distance_learning"] is None:
                result["distance_learning"] = value
            elif label == "origin date":
                result["origin_date"] = value
            elif "last revision" in label:
                result["last_revision_date"] = value

    return result


# Sanity check with a hand-rolled fragment.
_demo = """
<div id="main-col">
<table>
  <tr><th>Module title</th><td>Economic Principles</td></tr>
  <tr><th>Module code</th><td>BEE1029</td></tr>
  <tr><th>Credits</th><td>30</td></tr>
</table>
<table><tr><th>Number students taking module (anticipated)</th><td>450</td></tr></table>
<h2>Learning activities and teaching methods (given in hours of study time)</h2>
<table>
  <tr><th>Scheduled Learning and Teaching Activities</th><th>Guided independent study</th><th>Placement / study abroad</th></tr>
  <tr><td>61</td><td>239</td><td>0</td></tr>
</table>
<h2>Summative assessment (% of credit)</h2>
<table>
  <tr><th>Coursework</th><th>Written exams</th><th>Practical exams</th></tr>
  <tr><td>37</td><td>63</td><td>0</td></tr>
</table>
<h2>Details of summative assessment</h2>
<table>
  <tr><th>Form of assessment</th><th>% of credit</th></tr>
  <tr><td>Homework T1</td><td>10</td></tr>
  <tr><td>Homework T2</td><td>10</td></tr>
  <tr><td>Group work</td><td>17</td></tr>
  <tr><td>Midterm</td><td>17</td></tr>
  <tr><td>Final T1</td><td>23</td></tr>
  <tr><td>Final T2</td><td>23</td></tr>
</table>
<table><tr><th>NQF level (module)</th><td>4</td></tr></table>
</div>
"""
parse_module_page(_demo)

{'module_title': 'Economic Principles',
 'credits': 30,
 'num_students_anticipated': 450,
 'scheduled_hours': 61,
 'independent_hours': 239,
 'placement_hours': 0,
 'coursework_pct': 37,
 'written_exam_pct': 63,
 'practical_exam_pct': 0,
 'n_summative_items': 6,
 'nqf_level': 4,
 'distance_learning': None,
 'origin_date': None,
 'last_revision_date': None}

## 4. Fetch + save loop

The loop writes each successfully-parsed module to disk immediately (append mode), so even a keyboard-interrupt mid-run leaves us with every page scraped so far.

In [7]:
def fetch_module_html(module_code: str, academic_year: str) -> str | None:
    params = {"moduleCode": module_code, "ay": academic_year, "sys": "0"}
    try:
        r = requests.get(DETAIL_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    except requests.RequestException as e:
        print(f"  ! network error {module_code} {academic_year}: {e}")
        return None
    if r.status_code != 200:
        return None
    return r.text


def append_row(row: dict):
    """Append one row to the details CSV, writing a header if the file is new."""
    write_header = not DETAILS_PATH.exists()
    pd.DataFrame([row]).to_csv(DETAILS_PATH, mode="a", header=write_header, index=False)


FIELDS_FROM_INDEX = [
    "module_code", "academic_year", "dept_slug", "dept_name",
    "stage", "credits_raw", "term_raw",
]

print(f"Starting scrape of {len(todo):,} module pages...")
print("(output is appended to modules_details.csv as we go — safe to interrupt)\n")

for i, row in todo.iterrows():
    code = row["module_code"]
    year = row["academic_year"]

    html = fetch_module_html(code, year)
    if html is None:
        parsed = {k: None for k in [
            "module_title", "credits", "num_students_anticipated",
            "scheduled_hours", "independent_hours", "placement_hours",
            "coursework_pct", "written_exam_pct", "practical_exam_pct",
            "n_summative_items", "nqf_level", "distance_learning",
            "origin_date", "last_revision_date",
        ]}
        parsed["fetch_ok"] = False
    else:
        parsed = parse_module_page(html)
        parsed["fetch_ok"] = True

    out = {f: row[f] for f in FIELDS_FROM_INDEX if f in row}
    out.update(parsed)
    append_row(out)

    if (i + 1) % 25 == 0:
        print(f"  {i + 1:5d} / {len(todo):5d}  last: {code} {year}  "
              f"cw={parsed['coursework_pct']} exam={parsed['written_exam_pct']}")

    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\nDone. Output file:", DETAILS_PATH)

Starting scrape of 9,073 module pages...
(output is appended to modules_details.csv as we go — safe to interrupt)

     25 /  9073  last: ECM2102 2019/0  cw=None exam=None
     50 /  9073  last: ECM3156 2019/0  cw=None exam=None
     75 /  9073  last: ECMM124 2019/0  cw=None exam=None
    100 /  9073  last: ECMM168 2019/0  cw=None exam=None
    125 /  9073  last: EDUM052 2019/0  cw=100 exam=0
    150 /  9073  last: EFPM912 2019/0  cw=100 exam=0
    175 /  9073  last: PSY2303 2019/0  cw=50 exam=50
    200 /  9073  last: PSY3439 2019/0  cw=40 exam=60
    225 /  9073  last: PSYM210 2019/0  cw=90 exam=0
    250 /  9073  last: PYCM043 2019/0  cw=100 exam=0
    275 /  9073  last: PYCM089 2019/0  cw=100 exam=0
    300 /  9073  last: CLA2252 2019/0  cw=0 exam=100
    325 /  9073  last: CLA3114 2019/0  cw=30 exam=50
    350 /  9073  last: CLAM077 2019/0  cw=100 exam=0
    375 /  9073  last: BIO1333 2019/0  cw=33 exam=34
    400 /  9073  last: BIO3037 2019/0  cw=40 exam=60
    425 /  9073  last:

## 5. Quick sanity check on the output

In [9]:
details = pd.read_csv(DETAILS_PATH)
print(f"Total rows: {len(details):,}")
print(f"\nFetch success rate: {details['fetch_ok'].mean():.1%}")
print("\nCompleteness of key fields:")
for col in ["coursework_pct", "written_exam_pct", "practical_exam_pct",
            "scheduled_hours", "independent_hours", "num_students_anticipated",
            "n_summative_items", "nqf_level"]:
    pct = details[col].notna().mean() * 100
    print(f"  {col:30s}  {pct:5.1f}% non-null")

print("\nSample of 5 rows:")
details[[
    "module_code", "academic_year", "dept_name", "credits",
    "coursework_pct", "written_exam_pct", "practical_exam_pct",
    "scheduled_hours", "num_students_anticipated",
]].sample(min(5, len(details)))

Total rows: 9,073

Fetch success rate: 99.8%

Completeness of key fields:
  coursework_pct                   82.7% non-null
  written_exam_pct                 82.6% non-null
  practical_exam_pct               82.6% non-null
  scheduled_hours                  82.5% non-null
  independent_hours                82.3% non-null
  num_students_anticipated         79.8% non-null
  n_summative_items                82.5% non-null
  nqf_level                        82.6% non-null

Sample of 5 rows:


,module_code,academic_year,dept_name,credits,coursework_pct,written_exam_pct,practical_exam_pct,scheduled_hours,num_students_anticipated
9048,SOCM052,2024/5,Sociology,30.0,100.0,0.0,0.0,22.0,10.0
3684,SOC3040,2021/2,Sociology,30.0,100.0,0.0,0.0,11.0,60.0
2915,BIO1333,2021/2,Biosciences,15.0,33.0,34.0,33.0,49.0,380.0
9009,SOC1028,2024/5,Sociology,15.0,100.0,0.0,0.0,22.0,70.0
6580,BEE3053,2023/4,Economics,15.0,20.0,80.0,0.0,27.0,197.0


## Next step

Proceed to **`03_clean_module_data.ipynb`**, which cleans this raw scrape, derives per-module features (department code, level, faculty, contact ratio), and aggregates up to the department level ready for merging with NSS.
